## LangChain tools
Tools w LangChain to zewnętrzne funkcje lub usługi (np. wyszukiwarka, kalkulator, API), które agent może wywoływać podczas rozwiązywania zadań, aby rozszerzyć swoje możliwości poza samo generowanie tekstu.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/3_2_LangChain_tools.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Definiowanie narzędzi przez `@tool` i używanie ich przez agentów: własne narzędzia, wyszukiwarka Tavily i Wikipedia. |
| 🔑 **Klucze API** | `OPENAI_API_KEY`, `TAVILY_API_KEY` |
| 💲 **Koszt API** | niski – kilkanaście zapytań |
| ⏱️ **Czas wykonania** | < 30 s |

### Instalacja bibliotek

In [ ]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain langchain-core langchain-openai langchain-classic langchain-community langchain-tavily wikipedia python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.
# Dla wyszukiwania Tavily dodaj też TAVILY_API_KEY w Colab Secrets lub lokalnym pliku .env.
# from google.colab import userdata
# os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Model gotowy.")

Model gotowy.


### Proste narzędzie (kalkulator)

In [3]:
from langchain.tools import tool

@tool
def multiply_numbers(x: int, y: int) -> int:
    """Mnoży dwie liczby całkowite."""
    return x * y

print(multiply_numbers.invoke({"x": 6, "y": 7}))


42


### Użycie narzędzia z LLM

In [4]:
from langchain_classic.agents import initialize_agent, AgentType

tools = [multiply_numbers]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

# Model sam zdecyduje, że musi użyć multiply_numbers
agent.run("Ile wynosi dwukrotność liczby ludzi na Ziemi?")


/tmp/ipykernel_266843/4218389687.py:5: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = initialize_agent(
/tmp/ipykernel_266843/4218389687.py:13: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  agent.run("Ile wynosi dwukrotność liczby ludzi na Ziemi?")




> Entering new AgentExecutor chain...
Thought: Najpierw muszę ustalić aktualną liczbę ludzi na Ziemi. Szacuje się, że liczba ta wynosi około 8 miliardów. Następnie obliczę dwukrotność tej liczby.

Action:
```
{
  "action": "multiply_numbers",
  "action_input": {"x": 8000000000, "y": 2}
}
```

Observation: 16000000000
Thought:Dwukrotność liczby ludzi na Ziemi wynosi 16 miliardów. 

Action:
```
{
  "action": "Final Answer",
  "action_input": "Dwukrotność liczby ludzi na Ziemi wynosi 16 miliardów."
}
```


> Finished chain.


'Dwukrotność liczby ludzi na Ziemi wynosi 16 miliardów.'

### Kolejne narzędzie (prosta baza wiedzy)

In [5]:
@tool
def company_info(name: str) -> str:
    """Zwraca podstawowe informacje o firmie na podstawie nazwy."""
    companies = {
        "OpenAI": "Firma zajmująca się sztuczną inteligencją, twórca ChatGPT.",
        "LangChain": "Biblioteka ułatwiająca budowanie aplikacji z modelami językowymi.",
        "Tesla": "Producent samochodów elektrycznych i baterii."
    }
    return companies.get(name, "Nie znam tej firmy.")

tools = [company_info]
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

agent.run("Powiedz mi, czym zajmuje się firma Tesla.")




> Entering new AgentExecutor chain...
Muszę uzyskać podstawowe informacje o firmie Tesla, aby odpowiedzieć na pytanie dotyczące jej działalności. 
Action: company_info
Action Input: "Tesla"
Observation: Producent samochodów elektrycznych i baterii.
Thought:I now know the final answer
Final Answer: Firma Tesla zajmuje się produkcją samochodów elektrycznych i baterii.

> Finished chain.


'Firma Tesla zajmuje się produkcją samochodów elektrycznych i baterii.'

### Dostępne narzędzia LangChain
`TavilySearch` udostępnia kilka parametrów wejściowych. Starszy `ZeroShotAgent` używany przez `initialize_agent` wymaga narzędzi jednowejściowych, dlatego tutaj używamy aktualnego `create_agent`, który obsługuje schemat argumentów narzędzia.

Dokumentacja: [Tavily Search w LangChain](https://docs.langchain.com/oss/python/integrations/tools/tavily_search) oraz [agenci LangChain](https://docs.langchain.com/oss/python/langchain/agents).

In [6]:
import json
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# Inicjalizacja narzędzia wyszukiwarki
# klucz API do Tavily: https://app.tavily.com/home
search = TavilySearch(max_results=3)

# Bezpośrednie użycie narzędzia
query = "Najnowsze wiadomości o premierze Polski"
results = search.invoke({"query": query})

# Zależnie od wersji integracji wynik może być słownikiem, listą albo tekstem.
if isinstance(results, str):
    try:
        results = json.loads(results)
    except json.JSONDecodeError:
        results = [results]

if isinstance(results, dict):
    search_results = results.get("results", [])
elif isinstance(results, list):
    search_results = results
else:
    search_results = [results]

print(f"Wyniki wyszukiwania dla zapytania: '{query}'")
for i, result in enumerate(search_results, 1):
    if isinstance(result, dict):
        print()
        print(f"{i}. {result.get('title', 'Brak tytułu')}")
        print(f"   URL: {result.get('url', 'Brak URL')}")
        print(f"   Treść: {result.get('content', 'Brak treści')[:200]}...")
    else:
        print(f"{i}. {str(result)[:200]}")

# create_agent obsługuje narzędzia z wieloma polami argumentów.
agent_with_search = create_agent(model=llm, tools=[search])

# Przykładowe zapytanie wymagające wyszukiwania w internecie
search_result = agent_with_search.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who is the current Prime Minister of the UK and how old are they?",
            }
        ]
    }
)
response = search_result["messages"][-1].content
print(f"Agent answer: {response}")

Wyniki wyszukiwania dla zapytania: 'Najnowsze wiadomości o premierze Polski'

1. Donald tusk - Najnowsze informacje - WP Wiadomości
   URL: https://wiadomosci.wp.pl/tag/donald%20tusk
   Treść: Premier Donald Tusk poinformował, że Polska nie powtórzy błędów innych państw w kwestii migracji i zapowiedział bezwzględność w ochronie granic. Podkreślił w internetowym wpisie, że Polska realizuje w...

2. Donald Tusk - najnowsze informacje w TVN24
   URL: https://tvn24.pl/tagi/donald-tusk
   Treść: "Nie jesteście sami". Tusk z przekazem dla UkraińcówPolska

482 zdarzenia z uzasadnionym podejrzeniem popełnienia czynu zabronionegoPolska

Spór o Trybunał Konstytucyjny. Co wiadomo do tej poryPolska
...

3. Donald Tusk: najważniejsze informacje | Tygodnik Powszechny
   URL: https://www.tygodnikpowszechny.pl/donald-tusk-3253
   Treść: Premier Donald Tusk. Gdańsk, 20 kwietnia 2026 r. // Fot. Michał Ryniak / Agencja Wyborcza.pl

### Po wywiadzie Tuska dla „Financial Times”: czy pytanie o lojalność USA

In [7]:
import importlib
import wikipedia as wikipedia_client
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

# Wikimedia wymaga opisowego nagłówka User-Agent. Pakiet wikipedia domyślnie
# korzysta ze starego adresu HTTP, więc przełączamy go na endpoint HTTPS.
wikipedia_client.set_user_agent(
    "LangChainCourseNotebook/1.0 "
    "(https://github.com/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph)"
)

# inicjalizacja narzędzia Wikipedia
wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wikipedia_api = importlib.import_module("wikipedia.wikipedia")
wikipedia_api.API_URL = "https://en.wikipedia.org/w/api.php"

# bezpośrednie użycie
wiki_result = wikipedia.invoke("Python programming language")
print("Wikipedia - Python:")
print(f"{wiki_result[:300]}...")

# agent z dostępem do wielu źródeł informacji
tools_extended = [search, wikipedia, company_info]
agent_multi_tool = create_agent(model=llm, tools=tools_extended)

# zapytanie wymagające użycia różnych narzędzi
multi_tool_result = agent_multi_tool.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Znajdź informacje o Pythonie w Wikipedii, a następnie wyszukaj "
                    "najnowsze artykuły o nowych funkcjach w Python 3.13"
                ),
            }
        ]
    }
)
response = multi_tool_result["messages"][-1].content
print("Odpowiedź:")
print(response)

Wikipedia - Python:
Page: Python (programming language)
Summary: Python is a high-level, general-purpose programming language that emphasizes code readability, simplicity, and ease-of-writing with the use of significant indentation, an extensive ("batteries-included") standard library, and garbage collection. Python su...
Odpowiedź:
### Informacje o Pythonie

Python to wysokopoziomowy, ogólnego przeznaczenia język programowania, który kładzie nacisk na czytelność kodu, prostotę i łatwość pisania, wykorzystując znaczną wcięcia. Posiada rozbudowaną standardową bibliotekę oraz mechanizm zbierania śmieci. Python wspiera wiele paradygmatów programowania, ale szczególnie kładzie nacisk na programowanie obiektowe i dynamiczne typowanie.

Guido van Rossum rozpoczął prace nad Pythonem pod koniec lat 80. XX wieku jako następcą języka ABC. Wersja 3.0, wydana w 2008 roku, była dużą rewizją i nie była w pełni zgodna wstecz z wcześniejszymi wersjami. Od wersji 3.5 dodano możliwości i słowa kluczowe 